# Bonus A — RAG with Frameworks

**What you'll learn:** How LangChain and LlamaIndex fit into the RAG ecosystem — not as alternatives to each other, but as complementary tools that are frequently used together.

**Time:** ~45 minutes

---

## Why frameworks?

You've already built a working RAG system from scratch. So why would you use a framework?

- **Less boilerplate** for common patterns (loaders, splitters, chains)
- **Ecosystem** of integrations: swap ChromaDB for Pinecone, Ollama for OpenAI, with one line
- **Community** of examples, recipes, and debugging advice

The risk is that frameworks abstract away the details you now understand well. Having built the pipeline by hand, you're in the best possible position to use them: you can read through a LangChain chain or a LlamaIndex query engine and understand exactly what it's doing.

## LangChain and LlamaIndex: complementary, not competing

A common misconception is that you have to choose between LangChain and LlamaIndex. In practice they solve different problems and are frequently used together.

![LlamaIndex RAG architecture](../images/Llamaindex-Langchain.webp)

**LlamaIndex** is primarily a *data framework*. Its core abstraction is the **index**, a structure that ingests, chunks, embeds, and organises your documents for retrieval. It has many index types (vector, tree, keyword, knowledge graph) and handles heterogeneous data sources (PDFs, databases, APIs, Notion, Confluence) through a rich loader ecosystem. Think of it as the search engine layer of your RAG stack.

**LangChain** is primarily an *orchestration framework*. Its core abstraction is the **chain**, a composable sequence of steps that connects loaders, retrievers, prompts, LLMs, and output parsers. It excels at multi-step logic, agent tool use, and building pipelines where the retrieval backend can be swapped out. Think of it as the pipeline glue layer.

| | LangChain | LlamaIndex |
|---|---|---|
| Primary strength | Chain and agent orchestration | Data indexing and retrieval |
| Think of it as | The pipeline glue | The search engine |
| Core abstraction | Chain / Agent | Index / Query engine |
| Use when | You need multi-step logic, agents, tool use | You need sophisticated retrieval over many document types |
| Used together | As the outer orchestration layer | As the retrieval backend |

We will build the same RAG pipeline three times: first with LangChain alone, then with LlamaIndex alone, and finally with both working together (LlamaIndex handling the index and retrieval, LangChain handling the chain and prompt logic).

In [ ]:
# Sync framework dependencies for this notebook

!uv sync --extra bonus_a

---

## Part 1 - LangChain: orchestration and chaining

LangChain organises RAG around three concepts:

- **Document loaders**: read files into a standard `Document` format
- **Text splitters**: chunk documents with configurable size and overlap
- **Chains**: compose retrieval + prompt + generation into a single callable

The strength here is composability. Every step is a standard interface, so you can swap the vector store, the LLM, or the prompt without touching the rest of the pipeline. This is exactly the same pipeline you built in Module 4. LangChain just provides the interfaces so the pieces snap together.

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

from ragsst.parameters import DATA_PATH, EMBEDDING_MODEL, MODEL

In [ ]:
# 1. Load documents
loader = DirectoryLoader(DATA_PATH, glob='**/*.txt', loader_cls=TextLoader)
documents = loader.load()
print(f'Loaded {len(documents)} documents')

In [ ]:
# 2. Split into chunks
# RecursiveCharacterTextSplitter tries to split on paragraphs, then sentences, then words —
# similar in spirit to the split_text() function you wrote in Module 3.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)
chunks = splitter.split_documents(documents)
print(f'Created {len(chunks)} chunks')
print('\nSample chunk:')
print(chunks[0].page_content[:300])

In [ ]:
# 3. Embed and store in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='langchain_demo',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('Vector store ready.')

In [ ]:
# 4. Define a prompt template
# Compare this to the get_context_prompt() method in RAGTool - same idea, standard interface.
prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template=(
        'Use the following context to answer the question. '
        'Keep the answer concise.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}\n'
        'Answer:'
    )
)

In [ ]:
# 5. Build and run the chain
llm = Ollama(model=MODEL)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',           # 'stuff' = all chunks stuffed into one prompt
    retriever=retriever,
    chain_type_kwargs={'prompt': prompt_template},
    return_source_documents=True,
)

query = 'What services does the AI service center offer?'
result = qa_chain.invoke({'query': query})

print('Answer:')
print(result['result'])
print('\nSources:')
for doc in result['source_documents']:
    print(' -', doc.metadata.get('source', 'unknown'))

**Exercise:** Try changing `chain_type='stuff'` to `chain_type='map_reduce'`. This splits the chunks across multiple LLM calls and combines the answers - useful when the context is too large to fit in a single prompt. When would you prefer `map_reduce` over `stuff`?

Also try swapping `search_kwargs={'k': 3}` to `k=5` and see if the answer improves.

---

## Part 2 - LlamaIndex: data indexing and retrieval

LlamaIndex takes a different angle. Its central abstraction is the **index**, a data structure that knows how to store, organise, and retrieve your documents. Rather than wiring together individual steps, you describe your data and LlamaIndex figures out the retrieval strategy.

This makes it particularly powerful when your data is heterogeneous (PDFs, databases, APIs, structured tables) or when you need retrieval strategies beyond simple vector search (tree-based summarisation, keyword hybrid search, knowledge graphs).

The tradeoff: less explicit control over each step, but much less code for standard use cases.

In [ ]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama as LlamaOllama

In [ ]:
# Configure LlamaIndex to use our local models
Settings.llm = LlamaOllama(model=MODEL, request_timeout=120.0)
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL)
Settings.chunk_size = 500
Settings.chunk_overlap = 50

In [ ]:
# 1. Load documents
reader = SimpleDirectoryReader(DATA_PATH)
documents = reader.load_data()
print(f'Loaded {len(documents)} documents')

In [ ]:
# 2. Build the index
# from_documents() handles chunking, embedding, and indexing in one call.
# Compare this to the make_collection() method you wrote in Module 4.
index = VectorStoreIndex.from_documents(documents, show_progress=True)

In [ ]:
# 3. Create a query engine and ask a question
query_engine = index.as_query_engine(similarity_top_k=3)

query = 'What services does the AI service center offer?'
response = query_engine.query(query)

print('Answer:')
print(response)
print('\nSources:')
for node in response.source_nodes:
    print(f' - {node.metadata.get("file_name", "unknown")} (score: {node.score:.3f})')

**Exercise:** Try `index.as_chat_engine()` instead of `as_query_engine()`. This gives you a multi-turn conversation over your documents (ask a follow-up question that refers to the previous answer and see how it handles context).

Also try changing `VectorStoreIndex` to `SummaryIndex` (import it from `llama_index.core`). A summary index builds a tree over your documents and uses it to answer questions that require synthesising information across many chunks, rather than retrieving the top-k most similar ones. When would you prefer that over a vector index?

---

## Part 3 - Using them together

The most common production pattern is to use LlamaIndex for what it does best (indexing and retrieval) and LangChain for what it does best (orchestration, prompt management, chaining). LlamaIndex exposes its retrievers as LangChain-compatible objects, so the two plug together directly.

This gives you LlamaIndex's rich retrieval capabilities inside a LangChain pipeline - without having to choose between them.

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama

# LlamaIndex provides a LangChain-compatible retriever adapter
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama as LlamaOllama

from ragsst.parameters import DATA_PATH, EMBEDDING_MODEL, MODEL

In [ ]:
# 1. Build the LlamaIndex index (same as Part 2)
Settings.llm = LlamaOllama(model=MODEL, request_timeout=120.0)
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL)
Settings.chunk_size = 500
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader(DATA_PATH).load_data()
index = VectorStoreIndex.from_documents(documents, show_progress=True)
print('LlamaIndex index ready.')

In [ ]:
# 2. Expose the LlamaIndex retriever as a LangChain-compatible retriever
langchain_retriever = index.as_retriever(similarity_top_k=3).as_langchain_retriever()
print('LlamaIndex retriever wrapped for LangChain.')

In [ ]:
# 3. Build a LangChain chain around the LlamaIndex retriever
# LangChain handles the prompt template and chain logic;
# LlamaIndex handles the index and retrieval.
prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template=(
        'Use the following context to answer the question. '
        'Keep the answer concise.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}\n'
        'Answer:'
    )
)

llm = Ollama(model=MODEL)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=langchain_retriever,
    chain_type_kwargs={'prompt': prompt_template},
    return_source_documents=True,
)

query = 'What services does the AI service center offer?'
result = qa_chain.invoke({'query': query})

print('Answer:')
print(result['result'])
print('\nSources:')
for doc in result['source_documents']:
    print(' -', doc.metadata.get('source', 'unknown'))

The answer comes from LlamaIndex's retrieval engine, but the prompt construction and LLM call are managed by LangChain. Neither framework needs to know about the other's internals, they communicate through a standard retriever interface.

**Exercise:** Replace `VectorStoreIndex` with `SummaryIndex` in step 1 above and re-run. How does the answer change? The summary index builds a hierarchical summary tree over your documents and traverses it at query time, a fundamentally different retrieval strategy than nearest-neighbour vector search, but it plugs into the same LangChain chain without any changes to the orchestration code.

---

## Summary

| | Hand-built (Module 4) | LangChain | LlamaIndex | LangChain + LlamaIndex |
|---|---|---|---|---|
| Lines of code | ~60 | ~40 | ~20 | ~50 |
| Retrieval control | Full | Medium | High | High |
| Orchestration control | Full | High | Medium | High |
| Data source support | Manual | Many loaders | Many loaders | Many loaders |
| Best for | Learning, custom needs | Complex chains, agents | Sophisticated retrieval | Production systems |

The hand-built version from Module 4 is still the most important one: it's what taught you what every step does. The frameworks are tools that let you move faster once you understand the pipeline. And because you built it by hand, you'll never be confused by what a framework is doing under the hood.

---

## Further reading

- [LangChain RAG tutorial](https://python.langchain.com/docs/tutorials/rag/)
- [LlamaIndex starter tutorial](https://docs.llamaindex.ai/en/stable/getting_started/starter_example/)
- [Using LlamaIndex with LangChain](https://docs.llamaindex.ai/en/stable/community/integrations/using_with_langchain/)